# Query the BAFU-2026 dataset

Loads the parquet tables exported by `sentier-brightway files` (see issue [#36](https://github.com/Depart-de-Sentier/brightcon-2026-material/issues/36)) as pandas DataFrames, plus a DuckDB connection for SQL-style queries over the same files.

Data location: `input-data/bafu-2026/registry/` (relative to this notebook).

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

DATA_DIR = Path("input-data/bafu-2026")
REGISTRY_DIR = DATA_DIR / "registry"

assert REGISTRY_DIR.exists(), (
    f"{REGISTRY_DIR} not found - run `sentier-brightway files --out input-data/bafu-2026` first"
)

## Load the registry tables as DataFrames

In [2]:
processes = pd.read_parquet(REGISTRY_DIR / "processes.parquet")
biosphere = pd.read_parquet(REGISTRY_DIR / "biosphere.parquet")
exchanges = pd.read_parquet(REGISTRY_DIR / "exchanges.parquet")
methods = pd.read_parquet(REGISTRY_DIR / "methods.parquet")
cfs = pd.read_parquet(REGISTRY_DIR / "characterization-factors.parquet")

for name, df in [
    ("processes", processes),
    ("biosphere", biosphere),
    ("exchanges", exchanges),
    ("methods", methods),
    ("characterization-factors", cfs),
]:
    print(f"{name:26s} {df.shape[0]:>8,d} rows  x  {df.shape[1]} cols")

processes.head()

processes                    11,947 rows  x  8 cols
biosphere                    93,154 rows  x  7 cols
exchanges                   420,063 rows  x  13 cols
methods                          25 rows  x  4 cols
characterization-factors    276,704 rows  x  5 cols


,bw_id,database,code,name,reference_product,unit,location,production_amount
0,1,bafu-2026,0004e814-c18d-42e2-a3f7-ce1fa51a3c2c,"Radioactive waste, in final repository for nuc...","Radioactive waste, in final repository for nuc...",cubic meter,CH,1.0
1,2,bafu-2026,0016d799-566a-3a58-bf88-48e3a349ee00,"xxx Electricity, production mix MX","xxx Electricity, production mix MX",kilowatt hour,MX,1.0
2,3,bafu-2026,00173db7-7590-3ac7-bf27-a41684347176,"Acrylic varnish, 87.5% in H2O, at plant","Acrylic varnish, 87.5% in H2O, at plant",kilogram,RER,1.0
3,4,bafu-2026,0017b758-86e7-499e-b582-8cd4f2a1256d,"Disposal, glazing, 3-IV, U=0.6 W/m2K, 4 Low E ...","Disposal, glazing, 3-IV, U=0.6 W/m2K, 4 Low E ...",square meter,CH,1.0
4,5,bafu-2026,001835f5-ba6d-361a-8990-7c894d80c087,"Natural gas, liquefied, production AE, at frei...","Natural gas, liquefied, production AE, at frei...",normal cubic meter,KW,1.0


In [3]:
exchanges.head()

,process_bw_id,input_bw_id,input_database,input_code,type,amount,unit,uncertainty_type,loc,scale,minimum,maximum,negative
0,11934,11934,bafu-2026,ffcb318a-9215-3be9-ab3f-5c328dcc45a4,production,1.000000,kilogram,<NA>,NaN,NaN,NaN,NaN,<NA>
1,11934,2593,bafu-2026,37054425-8d95-384b-be96-a924dac9edc6,technosphere,0.000253,cubic meter,2,-8.282121,0.549306,NaN,NaN,<NA>
2,11934,15255,ef-3.1-biosphere,08a91e70-3ddc-11dd-9139-0050c2490048,biosphere,0.470000,megajoule,2,-0.755023,0.033829,NaN,NaN,<NA>
3,11934,9188,bafu-2026,c4a92617-9f99-3d7b-95c0-15fb110b80ad,technosphere,0.131000,kilowatt hour,2,-2.032558,0.033829,NaN,NaN,<NA>
4,11872,11872,bafu-2026,fe6e1850-9847-3fc5-b74f-dde093f435ac,production,1.000000,kilogram,<NA>,NaN,NaN,NaN,NaN,<NA>


## Optional: DuckDB connection for SQL over the same parquet files

Useful for larger/aggregate queries without loading a whole table into memory.

In [4]:
con = duckdb.connect()
processes_path = REGISTRY_DIR / "processes.parquet"
exchanges_path = REGISTRY_DIR / "exchanges.parquet"
con.execute(f"CREATE OR REPLACE VIEW processes AS SELECT * FROM read_parquet('{processes_path}')")
con.execute(f"CREATE OR REPLACE VIEW exchanges AS SELECT * FROM read_parquet('{exchanges_path}')")

con.sql("SELECT count(*) AS n_processes FROM processes")

┌─────────────┐
│ n_processes │
│    int64    │
├─────────────┤
│       11947 │
└─────────────┘

## Test cell: find electricity datasets

Sanity check that the data loaded correctly and is queryable: search `processes` for anything electricity-related, in both pandas and DuckDB, and confirm both agree.

In [5]:
# pandas: case-insensitive name match
electricity_pd = processes[processes["name"].str.contains("electricity", case=False, na=False)]

# duckdb: same filter via SQL, for cross-check
electricity_sql = con.sql(
    "SELECT bw_id, code, name, location FROM processes WHERE lower(name) LIKE '%electricity%'"
).df()

assert len(electricity_pd) > 0, "expected at least one electricity process"
assert len(electricity_pd) == len(electricity_sql), "pandas and duckdb filters disagree"

print(f"Found {len(electricity_pd):,d} electricity-related processes")
print(f"  {electricity_pd['location'].nunique()} distinct locations")

electricity_pd.sort_values("name").head(20)

Found 2,279 electricity-related processes
  97 distinct locations


,bw_id,database,code,name,reference_product,unit,location,production_amount
11120,11121,bafu-2026,edcd9e24-f73c-3e18-b9f7-465ef785d2e2,"Cogen unit 160kWe, common components for heat+...","Cogen unit 160kWe, common components for heat+...",unit,RER,1.0
3774,3775,bafu-2026,4f8b0a1a-11e4-3e56-887a-2ed03c0beb71,"Cogen unit 160kWe, components for electricity ...","Cogen unit 160kWe, components for electricity ...",unit,RER,1.0
10536,10537,bafu-2026,e0ef25e1-b21f-3735-9400-7c78de053f68,"Cogen unit 1MWe, common components for heat+el...","Cogen unit 1MWe, common components for heat+el...",unit,RER,1.0
9124,9125,bafu-2026,c358953c-2ee5-39fa-8997-ee192f630572,"Cogen unit 1MWe, components for electricity only","Cogen unit 1MWe, components for electricity only",unit,RER,1.0
4442,4443,bafu-2026,5e24cd07-20bf-31d2-b8f4-40bdf14cc4f9,"Cogen unit 200kWe diesel SCR, common component...","Cogen unit 200kWe diesel SCR, common component...",unit,RER,1.0
11578,11579,bafu-2026,f83267b1-e90a-3ed6-9c67-23024d78ffd5,"Cogen unit 200kWe diesel SCR, components for e...","Cogen unit 200kWe diesel SCR, components for e...",unit,RER,1.0
2924,2925,bafu-2026,3da8724d-c815-3f70-aa65-4079325d8fd1,"Cogen unit 200kWe, common components for heat+...","Cogen unit 200kWe, common components for heat+...",unit,RER,1.0
7698,7699,bafu-2026,a417512e-3012-33e1-ae0e-18a3165240a5,"Cogen unit 200kWe, components for electricity ...","Cogen unit 200kWe, components for electricity ...",unit,RER,1.0
3324,3325,bafu-2026,45c791f5-5a60-3adc-ae26-47483b103813,"Cogen unit 500kWe, common components for heat+...","Cogen unit 500kWe, common components for heat+...",unit,RER,1.0
118,119,bafu-2026,02a21fac-f948-31b4-957a-1c27fa00e408,"Cogen unit 500kWe, components for electricity ...","Cogen unit 500kWe, components for electricity ...",unit,RER,1.0


In [6]:
# CH grid-mix low-voltage electricity, used as the worked example in the sentier-brightway README
ch_grid = electricity_pd[
    (electricity_pd["location"] == "CH")
    & electricity_pd["name"].str.contains("low voltage.*production CH", case=False, regex=True)
]
ch_grid

,bw_id,database,code,name,reference_product,unit,location,production_amount
1181,1182,bafu-2026,19346bf2-f651-324a-904c-9b4f835d3d48,"Electricity, low voltage, production CH, at grid","Electricity, low voltage, production CH, at grid",kilowatt hour,CH,1.0


## Bonus: score a process with bw2calc (no Brightway project needed)

In [7]:
from sentier_brightway.datapackage import score

code = ch_grid.iloc[0]["code"]
result = score(DATA_DIR, code, "ef-3.1:climate-change")
print(f"{ch_grid.iloc[0]['name']!r} climate-change score: {result:.7f} kg CO2-eq per unit")

C:\Users\rusai1\PycharmProjects\trailrunner\database\.venv\lib\site-packages\bw2calc\__init__.py:59: UserWarning: No fast sparse solver found
  warnings.warn("No fast sparse solver found")


'Electricity, low voltage, production CH, at grid' climate-change score: 0.0320835 kg CO2-eq per unit